# Tweety-2c — Logique du premier ordre en C#/.NET (port natif IKVM)

> **Série Tweety — port C#/.NET natif (EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667)).**
> Ce notebook exploite la librairie Java [TweetyProject](https://tweetyproject.org/) **sans JVM** : les modules
> sont compilés vers du bytecode Java puis exécutés sur le runtime .NET via [IKVM](https://github.com/ikvm-refined/ikvm).
> Aucun `java` n'est requis à l'exécution — tout se passe dans le kernel `.net-csharp`.

Navigation : [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb) (logique propositionnelle) ·
[Tweety-2b-Semantics-Csharp](Tweety-2b-Semantics-Csharp.ipynb) (mondes possibles, conséquence sémantique) ·
**Tweety-2c-FOL (ce notebook)**.

---

## Objectifs pédagogiques

La [logique propositionnelle](Tweety-2-Basic-Logics-Csharp.ipynb) raisonne sur des **propositions entières**
(`Man_Socrates`, `Mortal_Socrates`) reliées par connecteurs. Elle ne peut ni exprimer la **généralité**
(« tous les hommes sont mortels ») ni le **lien interne** d'un énoncé (« Socrate est un homme »).

La **logique du premier ordre** (FOL, *first-order logic*) introduit :

| Concept | Rôle | Exemple |
|---------|------|---------|
| **Terme** (`Term`) | Objet du discours | la variable `X`, la constante `Socrate` |
| **Prédicat** (`Predicate`) | Propriété / relation sur les termes | `Homme(·)`, `Parent(·,·)` |
| **Atome** (`FolAtom`) | Énoncé atomique | `Homme(Socrate)` |
| **Quantificateur universel** `∀` | « pour tout » | `∀X (Homme(X) ⇒ Mortel(X))` |
| **Quantificateur existentiel** `∃` | « il existe » | `∃X Mortel(X)` |

Ce notebook montre comment **construire** des formules FOL de façon programmatique en C#,
puis **raisonner** dessus avec le raisonneur d'énumération de Tweety (`SimpleFolReasoner`) —
le célèbre **syllogisme** de la mortalité de Socrate, jusqu'à la **généralisation existentielle**.



## 1 — Runtime IKVM : exécuter Tweety sur .NET

On installe le runtime IKVM (machine virtuelle Java réimplémentée en .NET) à partir des paquets NuGet,
puis on pointe `IKVM.Home` vers une image fusionnée (base `any` + runtime spécifique `win-x64`).
La librairie `org.tweetyproject.tweety-pl.dll` (compilée côté build) se charge par `#r` ; elle embarque
**transitivement** le module `fol` (logique du premier ordre), aucun module supplémentaire à compiler ici.



In [1]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"



The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

In [2]:
using System.IO;
string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);
Console.WriteLine("IKVM home=" + (File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat")) ? "OK" : "MISSING"));



IKVM home=OK


In [3]:
#r "org.tweetyproject.tweety-pl.dll"



## 2 — Signature, termes et atomes

Une **signature** FOL fixe le vocabulaire : quels **prédicats** (avec leur arité) et quelles **constantes**
représentent les objets nommés. Les **variables** (`X`, `Y`) dénotent des objets arbitraires ; les **constantes**
(`Socrate`) dénotent un objet précis. Un **atome** applique un prédicat à des termes.

### 2.1 Construire un atome

On importe les espaces de noms Tweety puis on instancie un prédicat unaire et on l'applique à un terme.
L'API C# expose les classes Java en PascalCase mais conserve **les méthodes Java en minuscules**
(`.add`, `.size`) — c'est la convention de port IKVM.



In [4]:
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.fol.reasoner;
using org.tweetyproject.logics.fol.parser;

// Un prédicat unaire "Homme" et une variable X
var Homme = new Predicate("Homme", 1);
var X = new Variable("X");

// Atome Homme(X) : le prédicat appliqué à la variable
var hommeX = new FolAtom(Homme, X);
Console.WriteLine("Homme(X)        = " + hommeX);

// Une constante nommée Socrate
var socrate = new Constant("Socrate");
var hommeSocrate = new FolAtom(Homme, socrate);
Console.WriteLine("Homme(Socrate)  = " + hommeSocrate);



Homme(X)        = Homme(X)


Homme(Socrate)  = Homme(Socrate)


### 2.2 Atomes, prédicats et types de termes

`FolAtom` accepte une liste variable de termes (`Predicate`, `params Term[]`) — l'arité du prédicat
doit correspondre au nombre de termes fournis. `Term` est une **interface** (`...syntax.interfaces.Term`)
implémentée à la fois par `Variable` et `Constant` : un atome est donc polymorphe sur ses arguments.



In [5]:
// Prédicat binaire : Parent(·,·) — arité 2
var Parent = new Predicate("Parent", 2);
var anne = new Constant("Anne");
var bob = new Constant("Bob");
var parentAnneBob = new FolAtom(Parent, anne, bob);
Console.WriteLine("Parent(Anne,Bob) = " + parentAnneBob);

// Une variable peut figurer parmi les arguments (relation "Anne est parent de quelqu'un")
var Y = new Variable("Y");
var parentAnneY = new FolAtom(Parent, anne, Y);
Console.WriteLine("Parent(Anne,Y)   = " + parentAnneY);



Parent(Anne,Bob) = Parent(Anne,Bob)


Parent(Anne,Y)   = Parent(Anne,Y)


## 3 — Connecteurs et quantificateurs

Les connecteurs propositionnels (`Implication`, `Conjunction`, `Disjunction`, `Negation`) s'appliquent
aux formules FOL exactement comme en logique propositionnelle. La nouveauté est le **quantificateur**,
qui lie une variable à l'intérieur d'une formule :

- **`ForallQuantifiedFormula(formule, variable)`** → `∀X formule`
- **`ExistsQuantifiedFormula(formule, variable)`** → `∃X formule`

### 3.1 L'axiome classique : « tous les hommes sont mortels »

`∀X (Homme(X) ⇒ Mortel(X))` se construit en emboîtant un `Implication` dans un `ForallQuantifiedFormula`.



In [6]:
var Mortel = new Predicate("Mortel", 1);

// Homme(X) ⇒ Mortel(X)
var hommeX = new FolAtom(Homme, X);
var mortelX = new FolAtom(Mortel, X);
var impl = new Implication(hommeX, mortelX);
Console.WriteLine("Homme(X) => Mortel(X) = " + impl);

// ∀X (Homme(X) => Mortel(X))
var tousMortels = new ForallQuantifiedFormula(impl, X);
Console.WriteLine("∀X (Homme(X)=>Mortel(X)) = " + tousMortels);



Homme(X) => Mortel(X) = (Homme(X)=>Mortel(X))


∀X (Homme(X)=>Mortel(X)) = forall X: ((Homme(X)=>Mortel(X)))


### 3.2 Existential : « il existe un mortel »

`∃X Mortel(X)` ne quantifie qu'un atome. On réutilise le même atome `mortelX` (qui porte la variable `X`).



In [7]:
var existeMortel = new ExistsQuantifiedFormula(mortelX, X);
Console.WriteLine("∃X Mortel(X) = " + existeMortel);

// Négation d'une formule quantifiée : ¬∃X Mortel(X)  ("personne n'est mortel")
var personneMortel = new Negation(existeMortel);
Console.WriteLine("¬∃X Mortel(X) = " + personneMortel);



∃X Mortel(X) = exists X: (Mortel(X))


¬∃X Mortel(X) = !exists X: (Mortel(X))


## 4 — Raisonnement : le syllogisme

On rassemble les axiomes dans un **ensemble de croyances** (`FolBeliefSet`, équivalent FOL du
`PlBeliefSet` propositionnel) puis on interroge le raisonneur.

`SimpleFolReasoner` procède par **énumération des interprétations de Herbrand** : il construit
l'univers de Herbrand (les constantes apparaissant dans la base) et teste les formules sur les
interprétations possibles. C'est complet mais exponentiel — adapté à de petites bases pédagogiques.

> **Syllogisme classique.** Des prémisses `∀X (Homme(X) ⇒ Mortel(X))` et `Homme(Socrate)`,
> on infère `Mortel(Socrate)`. En termes de conséquence logique : **KB ⊨ Mortal(Socrates)**.



In [8]:
// Base de croyances : l'axiome universel + le fait Homme(Socrate)
var kb = new FolBeliefSet();
kb.add(tousMortels);
kb.add(new FolAtom(Homme, socrate));

Console.WriteLine("KB = " + kb);
Console.WriteLine("taille KB = " + kb.size());

var r = new SimpleFolReasoner();
var mortelSocrate = new FolAtom(Mortel, socrate);
Console.WriteLine("\nKB ⊨ Mortel(Socrate) ?  " + r.query(kb, mortelSocrate));



KB = { Homme(Socrate), forall X: ((Homme(X)=>Mortel(X))) }


taille KB = 2



KB ⊨ Mortel(Socrate) ?  true


### 4.1 Contrôle négatif : ce qui n'est **pas** impliqué

`Platon` n'apparaît dans aucune prémisse : `Homme(Platon)` n'est ni affirmé ni inférable.
Le raisonneur répond donc `false` à `KB ⊨ Mortel(Platon)` — la base est **sous-déterminée** pour Platon.
Ce `false` signifie « pas conséquence logique », **pas** « conséquence logique du contraire ».



In [9]:
var platon = new Constant("Platon");
var mortelPlaton = new FolAtom(Mortel, platon);
Console.WriteLine("KB ⊨ Mortel(Platon) ?  " + r.query(kb, mortelPlaton));

// Idem : Homme(Platon) n'est pas impliqué non plus
Console.WriteLine("KB ⊨ Homme(Platon) ?  " + r.query(kb, new FolAtom(Homme, platon)));



KB ⊨ Mortel(Platon) ?  false


KB ⊨ Homme(Platon) ?  false


### 4.2 Généralisation existentielle

Si `Mortel(Socrate)` est conséquence, alors `∃X Mortel(X)` l'est aussi : il **existe** au moins un mortel
(Socrate). C'est la règle d'**introduction du quantificateur existentiel**. Le raisonneur le confirme
directement sur la formule quantifiée.



In [10]:
Console.WriteLine("KB ⊨ ∃X Mortel(X) ?  " + r.query(kb, existeMortel));

// Symétriquement, ∀X Mortel(X) n'est PAS impliqué : seuls les hommes connus le sont
var tousMortels2 = new ForallQuantifiedFormula(mortelX, X);
Console.WriteLine("KB ⊨ ∀X Mortel(X) ?    " + r.query(kb, tousMortels2));



KB ⊨ ∃X Mortel(X) ?  true


KB ⊨ ∀X Mortel(X) ?    false


### 4.3 Une KB plus riche : raisonnement sur plusieurs faits

Ajoutons un second fait `Homme(Platon)` et un prédicat `Grec`. On vérifie que le syllogisme se propage
à tous les hommes connus, et qu'un énoncé mixte `∃X (Grec(X) ∧ Mortel(X))` devient conséquence dès
qu'au moins un Grec mortel est dans la base.



In [11]:
var Grec = new Predicate("Grec", 1);
var kb2 = new FolBeliefSet();
kb2.add(tousMortels);                       // ∀X (Homme(X) => Mortel(X))
kb2.add(new FolAtom(Homme, socrate));       // Homme(Socrate)
kb2.add(new FolAtom(Homme, platon));        // Homme(Platon)
kb2.add(new FolAtom(Grec, socrate));        // Grec(Socrate)

Console.WriteLine("KB2 ⊨ Mortel(Socrate) ? " + r.query(kb2, new FolAtom(Mortel, socrate)));
Console.WriteLine("KB2 ⊨ Mortel(Platon) ?  " + r.query(kb2, new FolAtom(Mortel, platon)));

// ∃X (Grec(X) ∧ Mortel(X)) : "il existe un Grec mortel"
var grecX = new FolAtom(Grec, X);
var existeGrecMortel = new ExistsQuantifiedFormula(
    new Conjunction(grecX, mortelX), X);
Console.WriteLine("KB2 ⊨ ∃X (Grec(X) ∧ Mortel(X)) ? " + r.query(kb2, existeGrecMortel));



KB2 ⊨ Mortel(Socrate) ? true


KB2 ⊨ Mortel(Platon) ?  true


KB2 ⊨ ∃X (Grec(X) ∧ Mortel(X)) ? true


---

## Exercices

> Les exercices sont à compléter. Conformément à la convention du dépôt, les stubs **ne lèvent jamais
> d'erreur** (`raise`/`throw` interdits) : ils retournent `null` / affichent un message. Le notebook
> s'exécute donc de bout en bout même non complété.



### Exercice 1 — Le chat mammifère

Construisez programmatiquement l'axiome `∀X (Chat(X) ⇒ Mammifère(X))`, ajoutez le fait `Chat(Felix)`,
puis vérifiez que `KB ⊨ Mammifère(Felix)` vaut `true`. Créez les prédicats `Chat` et `Mammifère` (arité 1)
et la constante `Felix`.



In [12]:
// TODO etudiant : construire la base et le raisonneur
// Indice : suivez la structure de la section 4 (Predicate, Constant, FolAtom, Implication,
//          ForallQuantifiedFormula, FolBeliefSet.add, SimpleFolReasoner.query).
// Etape 1 : déclarer les prédicats Chat / Mammifere et la constante Felix.
// Etape 2 : construire ∀X (Chat(X) => Mammifere(X)) et Chat(Felix).
// Etape 3 : interroger KB |= Mammifere(Felix).
bool? reponse = null;  // TODO etudiant : affecter le resultat de la requete
Console.WriteLine("KB ⊨ Mammifere(Felix) ? " + (reponse?.ToString() ?? "Exercice a completer"));



KB ⊨ Mammifere(Felix) ? Exercice a completer


### Exercice 2 — Transitivité : grand-parent

Avec un prédicat binaire `Parent(·,·)`, on veut définir « grand-parent » sans nouveau prédicat :
`∀X ∀Y ∀Z (Parent(X,Y) ∧ Parent(Y,Z) ⇒ GrandParent(X,Z))`. Construisez cette formule à partir de
deux quantificateurs universels emboîtés et affichez-la.

**Indice :** `ForallQuantifiedFormula` ne lie qu'**une** variable à la fois ; pour `∀X ∀Y ∀Z` il faut
emboîter **trois** constructeurs (le plus externe lie `Z`, l'intermédiaire lie `Y`, l'interne lie `X`).



In [13]:
// TODO etudiant : construire la formule GrandParent
// Indice : declarez Z = new Variable("Z"); construisez l'implication,
//          puis ForallQuantifiedFormula(... X) dans ForallQuantifiedFormula(... Y) dans ForallQuantifiedFormula(... Z).
object formuleGrandParent = null;  // TODO etudiant
Console.WriteLine("GrandParent = " + (formuleGrandParent?.ToString() ?? "Exercice a completer"));



GrandParent = Exercice a completer


### Exercice 3 — Détection d'inconsistance

Une base est **inconsistante** si elle implique une contradiction (ex. à la fois `P(a)` et `¬P(a)`).
Avec `SimpleFolReasoner`, une KB inconsistante implique **toute** formule (*ex falso quodlibet*).

Ajoutez à une KB les faits `Mortel(Socrate)` et `¬Mortel(Socrate)`, puis vérifiez que
`KB ⊨ Homme(Socrate)` vaut `true` (principe d'explosion) — bien qu'aucun axiome ne parle d'`Homme`.



In [14]:
// TODO etudiant : mettre en evidence le principe d'explosion
// Indice : kb.add(FolAtom Mortel(Socrate)) ; kb.add(new Negation(FolAtom Mortel(Socrate))).
//          Puis interrogez KB |= Homme(Socrate) sur un prédicat Homme non lie par vos axiomes.
bool? explosion = null;  // TODO etudiant
Console.WriteLine("KB inconsistante ⊨ Homme(Socrate) ? " + (explosion?.ToString() ?? "Exercice a completer"));



KB inconsistante ⊨ Homme(Socrate) ? Exercice a completer


---

## Conclusion

On a porté en C#/.NET natif (sans JVM) le raisonnement **du premier ordre** via Tweety/IKVM :

1. **Construction programmatique** des formules FOL — termes (`Variable`/`Constant`), prédicats (`Predicate`),
   atomes (`FolAtom`), connecteurs (`Implication`/`Conjunction`/`Negation`), quantificateurs
   (`ForallQuantifiedFormula`/`ExistsQuantifiedFormula`).
2. **Raisonnement sémantique** par énumération de Herbrand (`SimpleFolReasoner.query`) — le syllogisme
   `∀X (Homme(X) ⇒ Mortel(X)), Homme(Socrate) ⊨ Mortel(Socrate)`, le contrôle négatif, la généralisation
   existentielle, et une KB multi-faits.

### FOL vs propositionnel — ce que FOL apporte

| Aspect | Propositionnel (Tweety-2) | Premier ordre (ce notebook) |
|--------|---------------------------|------------------------------|
| Objets du discours | propositions atomiques entières | **termes** (variables + constantes) |
| Généralité | impossible | **`∀`** (pour tout) |
| Existence | impossible | **`∃`** (il existe) |
| Structure interne d'un énoncé | plate | prédicat appliqué à des termes |
| Raisonneur | tables de vérité / mondes | énumération de Herbrand |

### Références

- TweetyProject — [FOL module](https://tweetyproject.org/api/fol/) et [tutoriels](https://tweetyproject.org/).
- Port C#/.NET via IKVM — EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667).
- Notebook compagnon propositionnel : [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb).

